In [15]:
import pandas as pd

In [16]:
# Load dataset
df = pd.read_csv("../data/email_evaluation_dataset_theertha.csv")

# Display preview
display(df.head())

,email_text,expected_action,expected_tone
0,"Hi Theertha, this is a reminder about our team...",notify,neutral
1,Reminder: Project review meeting today at 4 PM...,notify,urgent
2,This is a gentle reminder for our scheduled ca...,notify,polite
3,Our weekly sync-up meeting has been scheduled ...,notify,neutral
4,Please remember to attend the client meeting t...,notify,urgent


In [17]:
df = df.head(100)
df.shape

(100, 3)

In [18]:

df.to_csv("../data/email_evaluation_dataset_theertha.csv", index=False)

In [19]:
import re

def clean_email_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)   # remove punctuation & numbers
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    return text

In [20]:

df['clean_text'] = df['email_text'].apply(clean_email_text)

In [21]:
print("Clean_text column created successfully!")
display(df[['email_text', 'clean_text']].head())

Clean_text column created successfully!


,email_text,clean_text
0,"Hi Theertha, this is a reminder about our team...",hi theertha this is a reminder about our team ...
1,Reminder: Project review meeting today at 4 PM...,reminder project review meeting today at pm pl...
2,This is a gentle reminder for our scheduled ca...,this is a gentle reminder for our scheduled ca...
3,Our weekly sync-up meeting has been scheduled ...,our weekly syncup meeting has been scheduled f...
4,Please remember to attend the client meeting t...,please remember to attend the client meeting t...


In [23]:
def email_assistant(email_text):
    email = email_text.lower()

    # Action rules
    if any(word in email for word in ["invoice", "payment", "submit", "confirm", "approve", "clarify", "share"]):
        action = "respond"
    elif any(word in email for word in ["reminder", "scheduled", "meeting", "orientation"]):
        action = "notify"
    elif any(word in email for word in ["newsletter", "promotion", "advertisement", "auto-reply"]):
        action = "ignore"
    else:
        action = "notify"

    # Tone rules
    if any(word in email for word in ["urgent", "immediately", "final", "overdue"]):
        tone = "urgent"
    elif any(word in email for word in ["please", "kindly", "congratulations"]):
        tone = "polite"
    else:
        tone = "neutral"

    return action, tone


### Apply function to dataset

In [24]:
df["prediction"] = df["email_text"].apply(email_assistant)
df.head()


,email_text,expected_action,expected_tone,clean_text,prediction
0,"Hi Theertha, this is a reminder about our team...",notify,neutral,hi theertha this is a reminder about our team ...,"(notify, neutral)"
1,Reminder: Project review meeting today at 4 PM...,notify,urgent,reminder project review meeting today at pm pl...,"(notify, polite)"
2,This is a gentle reminder for our scheduled ca...,notify,polite,this is a gentle reminder for our scheduled ca...,"(notify, neutral)"
3,Our weekly sync-up meeting has been scheduled ...,notify,neutral,our weekly syncup meeting has been scheduled f...,"(notify, neutral)"
4,Please remember to attend the client meeting t...,notify,urgent,please remember to attend the client meeting t...,"(notify, polite)"


### Split tuple into columns

In [25]:
df[["predicted_action", "predicted_tone"]] = pd.DataFrame(
    df["prediction"].tolist(), index=df.index
)

df.drop(columns=["prediction"], inplace=True)
df.head()

,email_text,expected_action,expected_tone,clean_text,predicted_action,predicted_tone
0,"Hi Theertha, this is a reminder about our team...",notify,neutral,hi theertha this is a reminder about our team ...,notify,neutral
1,Reminder: Project review meeting today at 4 PM...,notify,urgent,reminder project review meeting today at pm pl...,notify,polite
2,This is a gentle reminder for our scheduled ca...,notify,polite,this is a gentle reminder for our scheduled ca...,notify,neutral
3,Our weekly sync-up meeting has been scheduled ...,notify,neutral,our weekly syncup meeting has been scheduled f...,notify,neutral
4,Please remember to attend the client meeting t...,notify,urgent,please remember to attend the client meeting t...,notify,polite


### Compare with expected values

In [26]:
df["action_correct"] = df["predicted_action"] == df["expected_action"]
df["tone_correct"] = df["predicted_tone"] == df["expected_tone"]

df.head()


,email_text,expected_action,expected_tone,clean_text,predicted_action,predicted_tone,action_correct,tone_correct
0,"Hi Theertha, this is a reminder about our team...",notify,neutral,hi theertha this is a reminder about our team ...,notify,neutral,True,True
1,Reminder: Project review meeting today at 4 PM...,notify,urgent,reminder project review meeting today at pm pl...,notify,polite,True,False
2,This is a gentle reminder for our scheduled ca...,notify,polite,this is a gentle reminder for our scheduled ca...,notify,neutral,True,False
3,Our weekly sync-up meeting has been scheduled ...,notify,neutral,our weekly syncup meeting has been scheduled f...,notify,neutral,True,True
4,Please remember to attend the client meeting t...,notify,urgent,please remember to attend the client meeting t...,notify,polite,True,False


### Calculate accuracy

In [27]:
action_accuracy = df["action_correct"].mean() * 100
tone_accuracy = df["tone_correct"].mean() * 100

print(f"Action Accuracy: {action_accuracy:.2f}%")
print(f"Tone Accuracy: {tone_accuracy:.2f}%")


Action Accuracy: 76.00%
Tone Accuracy: 58.00%


### Error analysis-Action errors

In [28]:
action_errors = df[df["action_correct"] == False][
    ["email_text", "expected_action", "predicted_action"]
]

action_errors.head(10)


,email_text,expected_action,predicted_action
30,Please clear the pending amount at the earliest.,respond,notify
45,Reminder to upload your project report.,respond,notify
48,Academic documents required for verification.,respond,notify
54,Reminder to register for the exam.,respond,notify
61,Please let me know the project deadline.,respond,notify
63,Is the office open tomorrow?,respond,notify
64,Can you guide me on the internship process?,respond,notify
65,What is the submission format?,respond,notify
67,Can you help me reset my login credentials?,respond,notify
68,Who should I contact for further details?,respond,notify


### Error analysis-Tone errors

In [29]:
tone_errors = df[df["tone_correct"] == False][
    ["email_text", "expected_tone", "predicted_tone"]
]

tone_errors.head(10)


,email_text,expected_tone,predicted_tone
1,Reminder: Project review meeting today at 4 PM...,urgent,polite
2,This is a gentle reminder for our scheduled ca...,polite,neutral
4,Please remember to attend the client meeting t...,urgent,polite
6,Reminder to join the Zoom meeting at 3 PM today.,urgent,neutral
7,This email is to remind you of the academic re...,polite,neutral
8,Kind reminder about the mentor discussion sche...,polite,neutral
11,This is to remind you about the presentation r...,urgent,neutral
12,Please attend the planning meeting scheduled f...,neutral,polite
13,Reminder for today’s stand-up meeting at 9:30 AM.,urgent,neutral
15,Gentle reminder regarding tomorrow’s faculty m...,polite,neutral


### Save output CSV

In [ ]:
output_path = "../data/milestone2_output_theertha.csv"
df.to_csv(output_path, index=False)

Which type of emails were hardest to classify?

General query and notification emails were the hardest to classify because they often lack explicit keywords indicating whether a response is required or if the email is only informational.

Why did your rules fail in some cases?

The rule-based logic depends heavily on predefined keywords and simple patterns. It fails when emails use indirect language, overlapping intent, or when the same words appear in different contexts that imply different actions or tones.

How could an LLM improve this process?

An LLM can understand context and intent beyond keywords, allowing more accurate classification of nuanced emails.